# Retrieving data from Github

In [1]:
#Using the requests library to download the data from the evidently github repo
import requests

repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [2]:
#The response content in bytes
len(zip_response.content)

17545754

In [3]:
#Opening the file as a ZIP without saving to disk
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [4]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [5]:
#Reading one of the files
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [6]:
#Parsing metadata
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [7]:
#Removing file prefix
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [8]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [9]:
#A function to ingest/feed documentation into the system before indexing or querying it
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [10]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [11]:
!uv add gitsource

Resolved 119 packages in 6ms
Checked 116 packages in 20ms


In [12]:
#Downloading the same documentation with gitsource
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [13]:
document = files[10]

print(document.filename)
print(document.content[:160])

docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




In [14]:
document.parse()

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [15]:
#Preparing the documents for indexing
documents = [f.parse() for f in files]

In [16]:
len(documents)

95

# SEARCH

In [17]:
!uv add minsearch

Resolved 119 packages in 5ms
Checked 116 packages in 4ms


In [18]:
from minsearch import Index

In [19]:
#text_fields - fields to search with full-text search
#keyword_fields - fields for exact matching only
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [20]:
#Testing if its searching within the defined fields
query = 'LLM as a Judge'
results = index.search(query, num_results=5)

In [21]:
len(results)

5

In [49]:
print(results)

[{'start': 0, 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different contex

# CHUNKING

In [22]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [23]:
#Applying sliding window to our data
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [24]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [25]:
#Adding indexes to the chunks
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [26]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [27]:
document_chunks[2]

{'start': 1500,
 'content': 'ently v0.7.8">\n  ## **Evidently 0.7.8**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.8).\n</Update>\n\n<Update label="2025-06-04" description="Evidently v0.7.7">\n  ## **Evidently 0.7.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.7).\n</Update>\n\n<Update label="2025-05-25" description="Evidently v0.7.6">\n  ## **Evidently 0.7.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.6).\n</Update>\n\n<Update label="2025-05-09" description="Evidently v0.7.5">\n  ## **Evidently 0.7.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.5).\n</Update>\n\n<Update label="2025-05-05" description="Evidently v0.7.4">\n  ## **Evidently 0.7.4**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.4).\n</Update>\n\n<Update label="2025-04-25

In [28]:
#Creating an index for the chunks
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [29]:
results = chunk_index.search(query)

In [30]:
len(results)

10

In [31]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=3000, step=1500)

In [32]:
document_chunks[2]

{'start': 1500,
 'content': 'ently v0.7.8">\n  ## **Evidently 0.7.8**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.8).\n</Update>\n\n<Update label="2025-06-04" description="Evidently v0.7.7">\n  ## **Evidently 0.7.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.7).\n</Update>\n\n<Update label="2025-05-25" description="Evidently v0.7.6">\n  ## **Evidently 0.7.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.6).\n</Update>\n\n<Update label="2025-05-09" description="Evidently v0.7.5">\n  ## **Evidently 0.7.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.5).\n</Update>\n\n<Update label="2025-05-05" description="Evidently v0.7.4">\n  ## **Evidently 0.7.4**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.4).\n</Update>\n\n<Update label="2025-04-25

# RAG

In [33]:
from anthropic import Anthropic
anthropic_client = Anthropic()

In [34]:
search_result = chunk_index.search(query, num_results=5)

In [35]:
len(search_result)

5

In [36]:
print(search_result)

[{'start': 0, 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different contex

In [37]:
query = "How to implement LLM as a judge"

In [38]:
import json

search_result_json = json.dumps(search_result, indent=2)

In [39]:
print(search_result)

[{'start': 0, 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different contex

In [40]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [64]:
def llm(user_prompt, instructions=None, model='claude-haiku-4-5'):

    messages = []

    # if instructions is not None:
    #     messages.append({
    #         "role": "assistant",
    #         "content": instructions
    #     })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1000,
        system=instructions,
        messages=messages
    )

    return response.content

In [51]:
answer = llm(user_prompt, instructions)

In [52]:
print(answer)

When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:

* Just the inputs, or
* Both inputs and expected outputs (ground truth).

You can use this test dataset to:

* Run **experiments** and track if changes improve or degrade system performance.
* Run **regression testing** to ensure updates don’t break what was already working.
* **Stress-test** your system with complex or adversarial inputs to check its resilience.

![](/images/synthetic/synthetic_experiments_img.png)

You can create test datasets manually, collect them from real or historical data, or generate them synthetically. While real data is best, it is not always available or sufficient to cover all cases. Public LLM benchmarks help with general model comparisons but don’t reflect your specific use case. Manually writing test cases takes time and effort.

**Synthetic data helps here**. It’s especially useful when

In [59]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [60]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [61]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [66]:
rag("how do i setup alerts")

[TextBlock(citations=None, text='# Setting Up Alerts\n\nTo set up alerts in Evidently, follow these steps:\n\n## 1. **Access the Alerts Section**\n- Open your Project\n- Navigate to **"Alerts"** in the left menu\n\n## 2. **Configure Required Settings**\n\nYou must set up two things:\n\n### A. **Notification Channel**\nChoose where to receive alerts:\n- **Email** — Add email addresses\n- **Slack** — Add a Slack webhook\n- **Discord** — Add a Discord webhook\n\n### B. **Alert Condition**\nChoose what triggers an alert:\n\n#### **Option 1: Failed Tests**\nIf you use Tests (conditional checks) in your Project, you can toggle the option to alert on failed Tests in a Test Suite. Evidently will send an alert to your notification channel whenever any Test fails.\n\n**💡 Tip to reduce alert fatigue:** Use the `is_critical` parameter in your Tests. Set it to `False` to mark Tests as Warnings, which won\'t trigger alerts even if they fail.\n\n#### **Option 2: Custom Conditions**\nSet alerts on ind

In [65]:
rag('how do i customize my LLM judge')

[TextBlock(citations=None, text='# Customizing Your LLM Judge\n\nBased on the course materials, here\'s how to customize your LLM judge:\n\n## 1. **Create a Custom Prompt Template**\n\nDefine a custom evaluation criterion using a prompt template. Here\'s an example for evaluating verbosity:\n\n```python\nfrom evidently.llm.templates import BinaryClassificationPromptTemplate\n\nverbosity = BinaryClassificationPromptTemplate(\n    name="Verbosity",\n    description="Evaluate if the response is concise or verbose",\n    criteria="""Evaluate if the response is concise and to-the-point.\n    A concise response answers the question directly without unnecessary elaboration.\n    A verbose response adds extra information that wasn\'t asked for or explains things at length.""",\n    target_category="concise",\n    non_target_category="verbose",\n    uncertainty="unknown",\n    include_reasoning=True,\n    pre_messages=[("system", "You are an expert text evaluator. You will be given a text of th